# Data Pipeline & Low-Light Simulation
**Leather Defect Detection — EE7204/EC7205**

This notebook covers:
1. Downloading and exploring the **MVTec AD Leather** subset
2. **Low-light simulation** (gamma darkening, Gaussian noise, vignetting, uneven illumination)
3. **Preprocessing pipeline** (resize to 256×256, normalization, paired augmentation)
4. Generating **train / val / test splits** with low-light copies for defective images
5. Building a **`tf.data.Dataset`** factory for downstream model training
6. Verification with **PSNR / SSIM** degradation metrics

In [ ]:
# ── Install extra dependencies (Colab / fresh env) ────────────────────
%pip install -q kagglehub scikit-image tqdm
# opencv-python-headless is already present on Colab; install locally if needed
# %pip install -q opencv-python-headless

In [ ]:
import os, json, shutil, random, math, warnings
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric
import tensorflow as tf
warnings.filterwarnings('ignore')

print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs       : {len(gpus)} — {[g.name for g in gpus]}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION  —  edit paths here for your environment
# ══════════════════════════════════════════════════════════════════════
IS_COLAB    = os.path.exists('/content')
BASE_DIR    = Path('/content') if IS_COLAB else Path('.')

DATA_DIR      = BASE_DIR / 'mvtec_leather'
PROCESSED_DIR = BASE_DIR / 'processed_dataset'
LOGS_DIR      = PROCESSED_DIR / 'logs'

# ── Image dimensions ─────────────────────────────────────────────────
IMG_SIZE  = 256
CHANNELS  = 3
SEED      = 42

# ── Train / Val / Test ratio ─────────────────────────────────────────
TRAIN_R, VAL_R, TEST_R = 0.70, 0.15, 0.15

# ── Low-light simulation parameters ──────────────────────────────────
# Gamma > 1 darkens the image.
#   1.0       = original brightness (no change)
#   1.0–1.5   = subtle dim  (barely noticeable)
#   1.2–2.2   = mild/moderate dim  ← current setting (realistic factory floor)
#   2.5–4.0   = heavy dark (previous setting — was too dark)
GAMMA_MIN,  GAMMA_MAX  = 1.0,  1.3    # was (1.2, 2.2) -- reduced for DDPM clarity

NOISE_MIN,  NOISE_MAX  = 0.5,  3.0    # Gaussian noise std (0-255); was (2.0, 10.0)
VIG_MIN,    VIG_MAX    = 0.03, 0.12   # vignette strength; was (0.10, 0.35)

LL_COPIES_TRAIN = 3    # low-light copies per defective image
LL_COPIES_VAL   = 2
LL_COPIES_TEST  = 1

random.seed(SEED)
np.random.seed(SEED)

for split in ('train', 'val', 'test'):
    for sub in ('images', 'masks', 'lowlight'):
        (PROCESSED_DIR / split / sub).mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration:')
print(f'  Gamma range   : {GAMMA_MIN} – {GAMMA_MAX}  (1.0 = no darkening)')
print(f'  Noise std     : {NOISE_MIN} – {NOISE_MAX}')
print(f'  Vignette      : {VIG_MIN} – {VIG_MAX}')
print(f'  Uneven illum  : 0.05 – 0.20  (set in simulate_low_light)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DOWNLOAD  MVTec AD Leather subset
# ══════════════════════════════════════════════════════════════════════
"""
MVTec AD leather directory layout expected at DATA_DIR:
  train/good/             ~245 defect-free training images
  test/good/              defect-free test images
  test/{cut,fold,glue,poke,color}/   defective test images
  ground_truth/{cut,fold,glue,poke,color}/  binary masks (*_mask.png)

Manual download:  https://www.mvtec.com/company/research/datasets/mvtec-ad
  → extract the 'leather' folder to DATA_DIR
"""

def download_mvtec_leather(dst: Path) -> bool:
    if (dst / 'train' / 'good').exists():
        n = len(list((dst / 'train' / 'good').glob('*.png')))
        print(f'Dataset already present  ({n} train-good images).')
        return True
    try:
        import kagglehub
        src_root = Path(kagglehub.dataset_download('ipythonx/mvtec-ad'))
        # Locate the leather sub-folder
        candidates = [src_root / 'leather'] + list(src_root.rglob('leather'))
        leather_src = next((p for p in candidates if p.is_dir()), None)
        if leather_src:
            shutil.copytree(leather_src, dst)
            print(f'Dataset downloaded to {dst}')
            return True
    except Exception as e:
        print(f'kagglehub download failed: {e}')

    print('\nManual setup required:')
    print('  1. Download MVTec AD: https://www.mvtec.com/company/research/datasets/mvtec-ad')
    print(f'  2. Extract the leather/ folder to: {dst}')
    return False


dataset_ok = download_mvtec_leather(DATA_DIR)

# ── Explore structure ─────────────────────────────────────────────────
if dataset_ok:
    print('\nDataset contents:')
    for split in ('train', 'test'):
        split_d = DATA_DIR / split
        if split_d.exists():
            for cat in sorted(split_d.iterdir()):
                if cat.is_dir():
                    n = len(list(cat.glob('*.png')))
                    print(f'  {split}/{cat.name:<18}: {n:>4} images')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# LOW-LIGHT SIMULATION
# ══════════════════════════════════════════════════════════════════════

def gamma_correction(img: np.ndarray, gamma: float) -> np.ndarray:
    """Darkens image: gamma > 1 → darker (uint8 in, uint8 out)."""
    lut = np.array([(i / 255.0) ** (1.0 / gamma) * 255
                    for i in range(256)], dtype=np.uint8)
    return cv2.LUT(img, lut)


def add_gaussian_noise(img: np.ndarray, std: float) -> np.ndarray:
    """Zero-mean Gaussian sensor noise."""
    noise = np.random.normal(0.0, std, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def add_vignetting(img: np.ndarray, strength: float) -> np.ndarray:
    """Lens vignette: edges darker than centre."""
    h, w = img.shape[:2]
    sx = w / (2.0 * (1.0 + strength))
    sy = h / (2.0 * (1.0 + strength))
    Y, X = np.ogrid[:h, :w]
    mask = np.exp(-((X - w/2)**2 / (2*sx**2) +
                    (Y - h/2)**2 / (2*sy**2)))
    mask = (mask / mask.max()).astype(np.float32)[..., None]
    return (img.astype(np.float32) * mask).astype(np.uint8)


def add_uneven_illumination(img: np.ndarray, intensity: float) -> np.ndarray:
    """Directional illumination gradient (uneven factory lighting)."""
    h, w = img.shape[:2]
    angle = random.uniform(0, 2 * math.pi)
    Y, X  = np.mgrid[:h, :w]
    grad  = np.cos(angle) * X / w + np.sin(angle) * Y / h
    grad  = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
    factor = (1.0 - intensity + intensity * grad)[..., None]
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)


def simulate_low_light(
    img: np.ndarray,
    gamma: float = None,
    noise_std: float = None,
    vig_strength: float = None,
    rng_seed: int = None
) -> np.ndarray:
    """
    Full low-light degradation pipeline (all parameters from config cell).

    Steps:
      1. Gamma correction     GAMMA_MIN – GAMMA_MAX
      2. Uneven illumination  intensity 0.05 – 0.20
      3. Vignetting           VIG_MIN   – VIG_MAX
      4. Gaussian noise       NOISE_MIN – NOISE_MAX
    """
    if rng_seed is not None:
        random.seed(rng_seed)
        np.random.seed(rng_seed)

    gamma        = gamma        or random.uniform(GAMMA_MIN, GAMMA_MAX)
    noise_std    = noise_std    or random.uniform(NOISE_MIN, NOISE_MAX)
    vig_strength = vig_strength or random.uniform(VIG_MIN,   VIG_MAX)

    out = gamma_correction(img, gamma)
    out = add_uneven_illumination(out, random.uniform(0.02, 0.08))  # was 0.05–0.45
    out = add_vignetting(out, vig_strength)
    out = add_gaussian_noise(out, noise_std)
    return out


# ── Visualise all 4 levels using the updated parameter range ──────────
sample_paths = (list((DATA_DIR / 'test').glob('**/*.png'))
                if (DATA_DIR / 'test').exists() else [])
if sample_paths:
    demo_img = cv2.cvtColor(cv2.imread(str(sample_paths[0])), cv2.COLOR_BGR2RGB)
    demo_img = cv2.resize(demo_img, (IMG_SIZE, IMG_SIZE))

    configs = [
        (1.2,  2,  0.10, 'Very light  γ=1.2'),
        (1.5,  5,  0.20, 'Light       γ=1.5'),
        (1.8,  8,  0.28, 'Moderate    γ=1.8'),
        (2.2, 10,  0.35, 'Max         γ=2.2'),
    ]
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(demo_img); axes[0].set_title('Original'); axes[0].axis('off')
    for i, (g, n, v, title) in enumerate(configs):
        ll = simulate_low_light(demo_img, gamma=g, noise_std=n,
                                vig_strength=v, rng_seed=i)
        axes[i+1].imshow(ll)
        axes[i+1].set_title(title, fontsize=9)
        axes[i+1].axis('off')
    plt.suptitle('Low-Light Simulation — updated brighter range', fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(LOGS_DIR / 'll_demo.png'), dpi=120)
    plt.show()
else:
    print('No sample images found — run the download cell first.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PREPROCESSING  —  load, resize, normalise, augment
# ══════════════════════════════════════════════════════════════════════

def load_image(path: Path, size: int = IMG_SIZE) -> np.ndarray:
    """Load RGB image, resize to size×size, return float32 in [0, 1]."""
    img = cv2.imread(str(path))
    if img is None:
        raise IOError(f'Cannot read: {path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_LANCZOS4)
    return img.astype(np.float32) / 255.0


def load_mask(path: Path, size: int = IMG_SIZE) -> np.ndarray:
    """Load grayscale mask, resize (nearest-neighbour), binarise, return float32."""
    m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if m is None:
        raise IOError(f'Cannot read: {path}')
    m = cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST)
    _, m = cv2.threshold(m, 127, 1, cv2.THRESH_BINARY)
    return m.astype(np.float32)


def augment_pair(img: np.ndarray, mask: np.ndarray,
                 seed: int = None) -> tuple:
    """
    Consistent spatial augmentation for (image, mask) pairs:
      - Random horizontal flip
      - Random vertical flip
      - Random 90° rotation (k in {0,1,2,3})
      - Brightness jitter on image only
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    if random.random() > 0.5:
        img, mask = np.fliplr(img), np.fliplr(mask)
    if random.random() > 0.5:
        img, mask = np.flipud(img), np.flipud(mask)
    k = random.randint(0, 3)
    if k:
        img  = np.rot90(img,  k)
        mask = np.rot90(mask, k)
    if random.random() > 0.5:
        img = np.clip(img * random.uniform(0.8, 1.2), 0.0, 1.0)
    return img, mask

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# GATHER SAMPLES  +  STRATIFIED SPLIT
# ══════════════════════════════════════════════════════════════════════

DEFECT_TYPES = {'cut', 'fold', 'glue', 'poke', 'color'}


def gather_samples(data_dir: Path) -> list:
    """
    Walk the MVTec leather directory and collect dicts with keys:
      img       : Path to image (.png)
      mask      : Path to ground-truth mask or None
      dtype     : defect category name (str)
      defective : bool
      hint      : 'train' | 'test'  (used to seed the split)
    """
    samples = []

    # Training defect-free images
    train_good = data_dir / 'train' / 'good'
    if train_good.exists():
        for p in sorted(train_good.glob('*.png')):
            samples.append(dict(img=p, mask=None,
                                dtype='good', defective=False, hint='train'))

    # Test images (all categories)
    test_dir = data_dir / 'test'
    gt_dir   = data_dir / 'ground_truth'
    if test_dir.exists():
        for cat in sorted(test_dir.iterdir()):
            if not cat.is_dir():
                continue
            is_defective = cat.name != 'good'
            for p in sorted(cat.glob('*.png')):
                mask_p = None
                if is_defective and gt_dir.exists():
                    cand = gt_dir / cat.name / f'{p.stem}_mask.png'
                    mask_p = cand if cand.exists() else None
                samples.append(dict(img=p, mask=mask_p,
                                    dtype=cat.name,
                                    defective=is_defective,
                                    hint='test'))
    return samples


def stratified_split(samples: list, seed: int = SEED) -> tuple:
    """
    Stratified split by defect type.
    MVTec train/good samples all go to the training split.
    Test-category samples are split VAL_R : TEST_R.
    """
    by_type = defaultdict(list)
    for s in samples:
        by_type[s['dtype']].append(s)

    train, val, test = [], [], []
    for dtype, grp in by_type.items():
        random.seed(seed)
        random.shuffle(grp)

        tr_hint = [s for s in grp if s['hint'] == 'train']
        rest    = [s for s in grp if s['hint'] != 'train']

        denom  = VAL_R + TEST_R
        n_val  = max(1, round(len(rest) * VAL_R  / denom)) if rest else 0
        n_test = len(rest) - n_val

        train.extend(tr_hint)
        val.extend(rest[:n_val])
        test.extend(rest[n_val: n_val + n_test])

    return train, val, test


if dataset_ok:
    all_samples            = gather_samples(DATA_DIR)
    train_s, val_s, test_s = stratified_split(all_samples)

    def _count(lst):
        good = sum(1 for s in lst if not s['defective'])
        return len(lst), good, len(lst) - good

    tot_tr, g_tr, d_tr = _count(train_s)
    tot_vl, g_vl, d_vl = _count(val_s)
    tot_te, g_te, d_te = _count(test_s)

    print(f'Split summary:')
    print(f'  Train : {tot_tr:>4} total  (good={g_tr}, defective={d_tr})')
    print(f'  Val   : {tot_vl:>4} total  (good={g_vl}, defective={d_vl})')
    print(f'  Test  : {tot_te:>4} total  (good={g_te}, defective={d_te})')
else:
    all_samples = train_s = val_s = test_s = []
    print('Dataset not found — download first.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PROCESS  &  SAVE ALL SPLITS
# ══════════════════════════════════════════════════════════════════════

def process_and_save(
    samples: list,
    split: str,
    out_dir: Path,
    augment: bool = False,
    n_ll: int = 0
) -> list:
    """
    For each sample:
      • Save preprocessed image  →  {out_dir}/{split}/images/{stem}.npy   (float32 [0,1])
      • Save binary mask         →  {out_dir}/{split}/masks/{stem}.npy    (float32 {0,1})
      • Save n_ll low-light copies (defective only)
                                 →  {out_dir}/{split}/lowlight/{stem}_ll{k}.npy

    Returns list of metadata dicts.
    """
    img_d = out_dir / split / 'images'
    msk_d = out_dir / split / 'masks'
    ll_d  = out_dir / split / 'lowlight'
    for d in (img_d, msk_d, ll_d):
        d.mkdir(parents=True, exist_ok=True)

    meta = []
    for idx, s in enumerate(tqdm(samples, desc=f'  {split:>5}')):
        stem = f"{s['dtype']}_{s['img'].stem}"
        try:
            img = load_image(s['img'])
            if s['mask'] and s['mask'].exists():
                mask = load_mask(s['mask'])
            else:
                mask = np.zeros(img.shape[:2], dtype=np.float32)
        except Exception as e:
            print(f'    skip {s["img"].name}: {e}')
            continue

        if augment:
            img, mask = augment_pair(img, mask, seed=SEED + idx)

        np.save(str(img_d / f'{stem}.npy'),  img.astype(np.float32))
        np.save(str(msk_d / f'{stem}.npy'),  mask.astype(np.float32))

        rec = dict(stem=stem, dtype=s['dtype'],
                   defective=s['defective'], split=split, ll_names=[])

        if s['defective'] and n_ll > 0:
            img_u8 = (img * 255).astype(np.uint8)
            for k in range(n_ll):
                ll_img = simulate_low_light(img_u8,
                                           rng_seed=SEED + idx * 100 + k)
                ll_f32 = ll_img.astype(np.float32) / 255.0
                ll_name = f'{stem}_ll{k}'
                np.save(str(ll_d / f'{ll_name}.npy'), ll_f32)
                rec['ll_names'].append(ll_name)

        meta.append(rec)
    return meta


if train_s:
    print('Processing splits …')
    train_meta = process_and_save(train_s, 'train', PROCESSED_DIR,
                                  augment=True,  n_ll=LL_COPIES_TRAIN)
    val_meta   = process_and_save(val_s,   'val',   PROCESSED_DIR,
                                  augment=False, n_ll=LL_COPIES_VAL)
    test_meta  = process_and_save(test_s,  'test',  PROCESSED_DIR,
                                  augment=False, n_ll=LL_COPIES_TEST)

    all_meta = {'train': train_meta, 'val': val_meta, 'test': test_meta}
    with open(str(LOGS_DIR / 'metadata.json'), 'w') as f:
        json.dump(all_meta, f, indent=2, default=str)

    print('\nAll splits saved.')
else:
    print('No samples to process — check dataset download.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# tf.data DATASET FACTORY
# ══════════════════════════════════════════════════════════════════════

def build_tf_dataset(
    split: str,
    base: Path,
    batch_size: int = 8,
    shuffle: bool = True,
    include_ll: bool = True
) -> tuple:
    """
    Build a tf.data.Dataset that yields (image, mask) float32 batches.

    image : [B, IMG_SIZE, IMG_SIZE, 3]  in [0, 1]
    mask  : [B, IMG_SIZE, IMG_SIZE, 1]  binary {0, 1}

    When include_ll=True, low-light copies are added and matched to
    their corresponding ground-truth mask.

    Returns (dataset, n_samples).
    """
    img_d = base / split / 'images'
    msk_d = base / split / 'masks'
    ll_d  = base / split / 'lowlight'

    img_paths, msk_paths = [], []

    # Original images
    for ip in sorted(img_d.glob('*.npy')):
        mp = msk_d / ip.name
        if mp.exists():
            img_paths.append(str(ip))
            msk_paths.append(str(mp))

    # Low-light copies  →  match mask by stripping _ll{k} suffix
    if include_ll and ll_d.exists():
        for lp in sorted(ll_d.glob('*.npy')):
            # stem like: cut_000_ll0  →  base = cut_000
            parts     = lp.stem.rsplit('_ll', 1)
            base_stem = parts[0] if len(parts) == 2 else lp.stem
            mp = msk_d / f'{base_stem}.npy'
            if mp.exists():
                img_paths.append(str(lp))
                msk_paths.append(str(mp))

    n = len(img_paths)

    def _load_npy_pair(ip_bytes, mp_bytes):
        ip  = ip_bytes.numpy().decode()
        mp  = mp_bytes.numpy().decode()
        img = np.load(ip).astype(np.float32)          # [H, W, 3]
        msk = np.load(mp).astype(np.float32)[..., None]  # [H, W, 1]
        return img, msk

    def _tf_wrapper(ip, mp):
        img, msk = tf.numpy_function(
            _load_npy_pair, [ip, mp], [tf.float32, tf.float32])
        img.set_shape([IMG_SIZE, IMG_SIZE, CHANNELS])
        msk.set_shape([IMG_SIZE, IMG_SIZE, 1])
        return img, msk

    ds = tf.data.Dataset.from_tensor_slices((img_paths, msk_paths))
    if shuffle:
        ds = ds.shuffle(buffer_size=n, seed=SEED)
    ds = (ds
          .map(_tf_wrapper, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(batch_size)
          .prefetch(tf.data.AUTOTUNE))
    return ds, n


# ── Quick sanity check ────────────────────────────────────────────────
if (PROCESSED_DIR / 'train' / 'images').exists():
    train_ds, n_train = build_tf_dataset('train', PROCESSED_DIR, batch_size=4)
    val_ds,   n_val   = build_tf_dataset('val',   PROCESSED_DIR, batch_size=4)
    test_ds,  n_test  = build_tf_dataset('test',  PROCESSED_DIR, batch_size=4)

    print(f'tf.data datasets ready:')
    print(f'  train : {n_train} samples')
    print(f'  val   : {n_val}   samples')
    print(f'  test  : {n_test}  samples')

    for imgs, msks in train_ds.take(1):
        print(f'  Batch — imgs: {imgs.shape}  masks: {msks.shape}')
        print(f'  Image range: [{imgs.numpy().min():.3f}, {imgs.numpy().max():.3f}]')
        print(f'  Mask  unique values: {np.unique(msks.numpy())}')
else:
    print('Processed dataset not found — run processing cell first.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# VISUALISE SAMPLES  +  PSNR / SSIM DEGRADATION METRICS
# ══════════════════════════════════════════════════════════════════════

def visualise_and_measure(base: Path, split: str = 'train', n: int = 4) -> None:
    img_d = base / split / 'images'
    msk_d = base / split / 'masks'
    ll_d  = base / split / 'lowlight'

    # Prefer defective samples for visualisation
    all_files = sorted(img_d.glob('*.npy'))
    def_files  = [f for f in all_files if not f.stem.startswith('good_')]
    sel = (def_files[:n] if len(def_files) >= n else all_files[:n])
    if not sel:
        print('No samples to visualise.')
        return

    fig, axes = plt.subplots(len(sel), 3, figsize=(12, 4 * len(sel)))
    if len(sel) == 1:
        axes = axes[None, :]

    psnr_vals, ssim_vals = [], []

    for row, fp in enumerate(sel):
        img  = np.load(str(fp))
        msk_p = msk_d / fp.name
        msk  = np.load(str(msk_p)) if msk_p.exists() else np.zeros(img.shape[:2])

        # Low-light version: saved copy or generate on-the-fly
        ll_cands = sorted(ll_d.glob(f'{fp.stem}_ll*.npy'))
        if ll_cands:
            ll_img = np.load(str(ll_cands[0]))
        else:
            ll_img = simulate_low_light(
                (img * 255).astype(np.uint8)).astype(np.float32) / 255.0

        p = psnr_metric(img, ll_img, data_range=1.0)
        s = ssim_metric(img, ll_img, data_range=1.0, channel_axis=2)
        psnr_vals.append(p)
        ssim_vals.append(s)

        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f'Original\n{fp.stem[:28]}', fontsize=8)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(msk, cmap='gray', vmin=0, vmax=1)
        axes[row, 1].set_title('Ground-Truth Mask', fontsize=8)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(ll_img)
        axes[row, 2].set_title(
            f'Low-Light\nPSNR={p:.1f} dB  SSIM={s:.3f}', fontsize=8)
        axes[row, 2].axis('off')

    plt.suptitle(f'{split.upper()} split — original / mask / low-light',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(LOGS_DIR / f'samples_{split}.png'), dpi=120)
    plt.show()

    print(f'\nLow-light degradation quality (lower = more degraded):')
    print(f'  Mean PSNR  : {np.mean(psnr_vals):.2f} dB  (ideal original=inf)')
    print(f'  Mean SSIM  : {np.mean(ssim_vals):.4f}     (ideal original=1.0)')


if (PROCESSED_DIR / 'train' / 'images').exists():
    visualise_and_measure(PROCESSED_DIR, split='train', n=4)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FINAL DATASET SUMMARY
# ══════════════════════════════════════════════════════════════════════

def dataset_summary(base: Path) -> None:
    print('=' * 56)
    print('  Processed Dataset Summary')
    print('=' * 56)
    grand_total = 0
    for split in ('train', 'val', 'test'):
        img_files = list((base / split / 'images').glob('*.npy'))
        n_good = sum(1 for f in img_files if f.stem.startswith('good_'))
        n_def  = len(img_files) - n_good
        n_ll   = len(list((base / split / 'lowlight').glob('*.npy')))
        total  = len(img_files) + n_ll
        grand_total += total
        print(f'\n  {split.upper()}:')
        print(f'    defect-free originals : {n_good:>4}')
        print(f'    defective  originals  : {n_def:>4}')
        print(f'    low-light  copies     : {n_ll:>4}')
        print(f'    ─────────────────────────────')
        print(f'    total samples         : {total:>4}')
    print(f'\n  GRAND TOTAL               : {grand_total}')
    print('=' * 56)
    print('\nFiles stored as float32 .npy arrays:')
    print(f'  images   : [{base}/{{split}}/images/]    shape ({IMG_SIZE},{IMG_SIZE},3)')
    print(f'  masks    : [{base}/{{split}}/masks/]     shape ({IMG_SIZE},{IMG_SIZE})')
    print(f'  lowlight : [{base}/{{split}}/lowlight/]  shape ({IMG_SIZE},{IMG_SIZE},3)')


dataset_summary(PROCESSED_DIR)
print('\nData pipeline complete. Proceed to Synthetic_image_generation.ipynb')
print('  and Leather_detection_segmentation.ipynb for the next stages.')